# Ames Mutagenicity: Classical and Graph Neural Network Models

## Part 8 — Model-free structural baselines

A ranking of atoms by degree agrees with established toxicophores at $1.214\times$ chance on
the primary partition, against $1.242\times$ for a trained graph network's Integrated
Gradients and $1.116\times$ for an untrained one. This notebook establishes whether that
result holds, what produces it, and how much of the reported uncertainty it survives.

Nothing here is trained. The one model used is the representative GIN saved in Part 2.

**Contents**

1. Environment
2. Pre-registration
3. Data and partitions
4. Evaluation sets and scoring
5. Structural rankings across alert regimes
6. Replication on two further partitions
7. Mechanism: what the trained model and the degree ranking have in common
8. Agreement across retained fractions
9. Control distributions
10. What predicts the effect of training
11. Uncertainty on three axes
12. Dependence between partitions
13. Summary

*Approximate runtime: two to three hours on CPU; under an hour if a GPU is present.*

## 1. Environment

In [1]:
!pip install -q rdkit
!pip install -q statsmodels
!pip install -q torch_geometric

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 46.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 925.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 7.0 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = "/content/drive/MyDrive/ames_pipeline"
DATADIR, R, EXPL, MODELS = f"{WORK}/data", f"{WORK}/results", f"{WORK}/explain", f"{WORK}/models"
for d in (R, EXPL): os.makedirs(d, exist_ok=True)
print("Results:", R)

Mounted at /content/drive
Results: /content/drive/MyDrive/ames_pipeline/results


## 2. Pre-registration

The decisions below are fixed before any result in this notebook is read. They are recorded
here so that the reading of the results cannot be shaped by the results.

**The primary baseline is atom degree.** Ten further structural rankings are computed and
reported as a sensitivity panel, not as a menu from which the strongest is selected after the
fact. Taking the maximum of a panel as the reference would bias the comparison in the
direction of this paper's own critical conclusion, which is the direction a reader should
trust least. Where the maximum differs from degree it is reported as such, with the direction
of the selection bias stated.

Degree is chosen because it is the simplest ranking available, because it requires no model
and no training, and because a mechanism for it is already proposed: the dominant alert in
the library matches fused aromatic systems, whose atoms are by construction of high degree.

**Should degree fall behind the trained model on the replication partitions**, that is not
grounds to change the baseline. It triggers the fourth framing: the effect is
partition-dependent, and the claim weakens accordingly.

**Three readings of the missing cell of the two-by-two design** (Integrated Gradients on a
multilayer perceptron over circular fingerprints, run separately in Part 9) are fixed now:

| result | reading |
|---|---|
| close to the graph model's $1.23$ | the difficulty lies with the gradient rule, not the graph representation |
| close to TreeSHAP's $1.54$ | the difficulty lies with the graph representation and message passing |
| between the two | the two contributions do not separate; the claim is split between them |

**Terminology.** These rankings are called *model-free structural baselines*, not null
distributions. Atom degree measures substitution and ring fusion, both of which are chemical
information; calling it a null would misdescribe it and would contradict the mechanism
proposed above. The claim is that the ranking needs no model, no training and no attribution
method, which is what makes it a reference point.

## 3. Data and partitions

Bemis--Murcko scaffolds are computed once for the whole set and reused, so that the thirty
partitions below cost thirty groupings rather than thirty scaffold perceptions. The primary
partition is reconstructed and checked against the file written in Part 2; a mismatch would
invalidate every comparison in this notebook.

In [3]:
import numpy as np, pandas as pd, json, time
from collections import defaultdict
from rdkit import Chem, RDLogger
from rdkit.Chem.Scaffolds import MurckoScaffold
RDLogger.DisableLog("rdApp.*")

SEED = 42
PRIMARY_SPLIT_SEED = 101
REPLICATE_SEEDS = [201, 202]
BENCH_SEEDS = list(range(42, 57))
K_FRAC = 0.30
K_GRID = [0.10, 0.20, 0.30, 0.40, 0.50]
N_TIEBREAK = 200
N_UNTRAINED = 500
N_BOOT = 10000

df = pd.concat([pd.read_csv(f"{DATADIR}/{f}") for f in ["train.csv", "valid.csv", "test.csv"]],
               ignore_index=True)[["Drug_ID", "smiles", "label"]]
df["canonical"] = df["smiles"].apply(
    lambda s: Chem.MolToSmiles(Chem.MolFromSmiles(s)) if Chem.MolFromSmiles(s) else None)
df = df.dropna(subset=["canonical"]).drop_duplicates(
    subset="canonical", keep="first").reset_index(drop=True)

def _scaffold(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return ""
    try:
        return MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
    except Exception:
        try:
            return Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(mol))
        except Exception:
            return smi

t0 = time.time()
SCAF = df["canonical"].map(_scaffold).values
print(f"{len(df)} compounds, {len(set(SCAF))} distinct scaffolds "
      f"({time.time()-t0:.0f} s)")

def scaffold_split(seed, frac=(0.8, 0.1, 0.1), order="size"):
    """order='size' reproduces the splitter used throughout; 'random' shuffles groups."""
    groups = defaultdict(list)
    for i, s in enumerate(SCAF):
        groups[s].append(df.index[i])
    gs = list(groups.values())
    rng = np.random.default_rng(seed)
    if order == "size":
        gs.sort(key=lambda g: (-len(g), rng.random()))
    else:
        rng.shuffle(gs)
    n = len(df); a, b = int(frac[0]*n), int((frac[0]+frac[1])*n)
    tr, va, te = [], [], []
    for g in gs:
        if len(tr) + len(g) <= a:              tr += g
        elif len(tr) + len(va) + len(g) <= b:  va += g
        else:                                   te += g
    return df.loc[tr], df.loc[va], df.loc[te]

def random_split(seed, frac=(0.8, 0.1, 0.1)):
    rng = np.random.default_rng(seed)
    idx = rng.permutation(df.index.values)
    n = len(df); a, b = int(frac[0]*n), int((frac[0]+frac[1])*n)
    return df.loc[idx[:a]], df.loc[idx[a:b]], df.loc[idx[b:]]

_, _, te_primary = scaffold_split(PRIMARY_SPLIT_SEED)
ref = pd.read_csv(f"{R}/primary_test.csv")
same = set(te_primary["canonical"]) == set(ref["canonical"])
print(f"primary partition reproduces primary_test.csv: {same} "
      f"({len(te_primary)} vs {len(ref)} compounds)")
assert same, "split reconstruction does not match Part 2; stop here"

7255 compounds, 1577 distinct scaffolds (3 s)
primary partition reproduces primary_test.csv: True (726 vs 726 compounds)


## 4. Evaluation sets and scoring

The evaluation set and the agreement measure follow the attribution analysis exactly: the
overlap between the highest-ranked 30% of atoms and the toxicophore atoms, divided by the
chance overlap, aggregated as a ratio of means. Ties are broken at random and every ranking
is scored over 200 independent tie-breaks, because several of the descriptors are binary and
leave large numbers of ties.

Every quantity that depends on the alert library is recomputed whenever that library changes:
the evaluation set, the chance level, and the attainable maximum. The attainable maximum is
measured from a ranking that places every toxicophore atom first, since a molecule with fewer
alert atoms than the selection retains cannot reach the nominal bound.

In [4]:
TOXICOPHORES = {
    "aromatic_nitro":         "[a][N+](=O)[O-]",
    "aromatic_amine":         "[a][NX3;H2,H1;!$(N[!#6]);!$(N=*)]",
    "aromatic_nitroso":       "[a][NX2]=[OX1]",
    "alkyl_nitrite":          "[CX4][OX2][NX2]=[OX1]",
    "nitrosamine":            "[NX3][NX2]=[OX1]",
    "epoxide":                "C1OC1",
    "aziridine":              "C1NC1",
    "azide":                  "[NX2]=[NX2+]=[NX1-]",
    "azo_aromatic":           "[a][NX2]=[NX2][a]",
    "aliphatic_halide":       "[CX4][F,Cl,Br,I]",
    "hetero_hetero_bond":     "[N,O;H1,H2][N,O;H1,H2]",
    "polycyclic_aromatic":    "c1ccc2ccccc2c1",
    "aromatic_hydroxylamine": "[a][NX3;H1][OX2;H1]",
    "michael_acceptor":       "[CX3]=[CX3][CX3]=[OX1]",
    "aromatic_azoxy":         "[a][NX2]=[NX3+]([O-])[a]",
    "nitrogen_mustard":       "[NX3]([CX4][CX4][F,Cl,Br,I])[CX4][CX4][F,Cl,Br,I]",
    "aromatic_hydrazine":     "[a][NX3;H1][NX3;H2]",
    "alkyl_sulfonate":        "[CX4][OX2][SX4](=[OX1])(=[OX1])",
}
PATTERNS = {k: Chem.MolFromSmarts(v) for k, v in TOXICOPHORES.items()}
assert all(p is not None for p in PATTERNS.values()), "a SMARTS pattern failed to parse"

REGIMES = {
    "full":            list(TOXICOPHORES),
    "no_polycyclic":  [k for k in TOXICOPHORES if k != "polycyclic_aromatic"],
    "polycyclic_only": ["polycyclic_aromatic"],
}

def build_eval(frame, alerts):
    pats = {k: PATTERNS[k] for k in alerts}
    ev, n_none, n_all = [], 0, 0
    for smi, lab in zip(frame["canonical"], frame["label"]):
        if lab != 1:
            continue
        mol = Chem.MolFromSmiles(smi)
        if mol is None or mol.GetNumAtoms() == 0:
            continue
        tox = set()
        for p in pats.values():
            for m in mol.GetSubstructMatches(p):
                tox.update(m)
        if not tox:
            n_none += 1; continue
        if len(tox) == mol.GetNumAtoms():
            n_all += 1; continue
        ev.append({"smiles": smi, "mol": mol, "tox": tox, "n_atoms": mol.GetNumAtoms()})
    chance = np.array([len(e["tox"]) / e["n_atoms"] for e in ev])
    return ev, chance, n_none, n_all

def overlap(ev, scores, k_frac, tie_seed):
    rng = np.random.default_rng(tie_seed)
    out = np.full(len(ev), np.nan)
    for i, (e, sc) in enumerate(zip(ev, scores)):
        if sc is None or len(sc) != e["n_atoms"]:
            continue
        n = e["n_atoms"]; k = max(1, int(round(k_frac * n)))
        order = np.lexsort((rng.random(n), -np.asarray(sc, dtype=float)))
        out[i] = len(set(order[:k].tolist()) & e["tox"]) / k
    return out

def lift(ov, chance):
    m = ~np.isnan(ov)
    return float(ov[m].mean() / chance[m].mean()) if m.sum() else np.nan

def ceiling(ev, chance, k_frac):
    oracle = [np.array([1.0 if a in e["tox"] else 0.0 for a in range(e["n_atoms"])])
              for e in ev]
    return lift(overlap(ev, oracle, k_frac, 0), chance)

def scored(ev, scores, k_frac=K_FRAC, n_draws=N_TIEBREAK, base=0):
    v = np.array([lift(overlap(ev, scores, k_frac, base + j), chance_of(ev))
                  for j in range(n_draws)])
    return v

CHANCE_CACHE = {}
def chance_of(ev):
    key = id(ev)
    if key not in CHANCE_CACHE:
        CHANCE_CACHE[key] = np.array([len(e["tox"]) / e["n_atoms"] for e in ev])
    return CHANCE_CACHE[key]

EVAL, CHANCE, N_NONE, N_ALL = build_eval(te_primary, REGIMES["full"])
CHANCE_CACHE[id(EVAL)] = CHANCE
print(f"mutagenic test compounds : {int(te_primary.label.sum())}")
print(f"  no alert matched       : {N_NONE}")
print(f"  matched in entirety    : {N_ALL}")
print(f"evaluation set           : {len(EVAL)}")
print(f"mean chance overlap      : {CHANCE.mean():.4f}")
print(f"attainable maximum, k=.30: {ceiling(EVAL, CHANCE, K_FRAC):.4f}")

mutagenic test compounds : 336
  no alert matched       : 86
  matched in entirety    : 13
evaluation set           : 237
mean chance overlap      : 0.4360
attainable maximum, k=.30: 1.9144


## 5. Structural rankings across alert regimes

Eleven rankings computed from the molecular graph alone. The alert library contains one
pattern, matching fused aromatic systems, that is composed entirely of carbon; the remaining
seventeen are centred on heteroatoms. A single agreement figure therefore averages two
regimes that reward opposite rankings, so each ranking is scored separately under the full
library, with that pattern removed, and with that pattern alone.

In [5]:
HETERO = {"N","O","S","P","F","Cl","Br","I","Se","B","Si"}

def descriptor(ev, name):
    out = []
    for e in ev:
        ats = list(e["mol"].GetAtoms())
        if   name == "degree":            v = [a.GetDegree() for a in ats]
        elif name == "heteroatom":        v = [1.0 if a.GetSymbol() in HETERO else 0.0 for a in ats]
        elif name == "hetero_neighbours": v = [sum(1 for nb in a.GetNeighbors()
                                                   if nb.GetSymbol() in HETERO) for a in ats]
        elif name == "aromatic":          v = [1.0 if a.GetIsAromatic() else 0.0 for a in ats]
        elif name == "in_ring":           v = [1.0 if a.IsInRing() else 0.0 for a in ats]
        elif name == "abs_formal_charge": v = [abs(a.GetFormalCharge()) for a in ats]
        elif name == "hydrogens":         v = [a.GetTotalNumHs() for a in ats]
        elif name == "hetero_plus_degree":
            v = [(2.0 if a.GetSymbol() in HETERO else 0.0) + 0.25 * a.GetDegree() for a in ats]
        elif name == "hetero_or_neighbour":
            v = [1.0 if (a.GetSymbol() in HETERO or
                 any(nb.GetSymbol() in HETERO for nb in a.GetNeighbors())) else 0.0
                 for a in ats]
        elif name == "rarity":
            key = [(a.GetSymbol(), a.GetDegree(), a.GetIsAromatic()) for a in ats]
            cnt = {k: key.count(k) for k in set(key)}
            v = [1.0 / cnt[k] for k in key]
        elif name == "degree_deviation":
            d = np.array([a.GetDegree() for a in ats], dtype=float)
            v = list(np.abs(d - d.mean()))
        else:
            raise ValueError(name)
        out.append(np.asarray(v, dtype=float))
    return out

RANKINGS = ["degree", "hetero_neighbours", "degree_deviation", "hetero_plus_degree",
            "abs_formal_charge", "heteroatom", "hetero_or_neighbour", "rarity",
            "aromatic", "in_ring", "hydrogens"]

rows = []
for reg, alerts in REGIMES.items():
    ev, ch, nn, na = build_eval(te_primary, alerts)
    CHANCE_CACHE[id(ev)] = ch
    ceil = ceiling(ev, ch, K_FRAC)
    for name in RANKINGS:
        v = scored(ev, descriptor(ev, name), base=1000)
        rows.append({"regime": reg, "ranking": name, "n_eval": len(ev),
                     "chance": ch.mean(), "ceiling": ceil,
                     "lift": v.mean(), "lift_sd": v.std(ddof=1),
                     "pct_of_ceiling": 100 * (v.mean() - 1) / (ceil - 1)})
    rnd = np.array([lift(overlap(ev, [np.random.default_rng(7000 + 13*j + i).random(e["n_atoms"])
                                      for i, e in enumerate(ev)], K_FRAC, 7000 + j), ch)
                    for j in range(N_TIEBREAK)])
    rows.append({"regime": reg, "ranking": "random atom order", "n_eval": len(ev),
                 "chance": ch.mean(), "ceiling": ceil,
                 "lift": rnd.mean(), "lift_sd": rnd.std(ddof=1),
                 "pct_of_ceiling": 100 * (rnd.mean() - 1) / (ceil - 1)})
    print(f"{reg:16s} n={len(ev):3d}  chance={ch.mean():.4f}  ceiling={ceil:.4f}  "
          f"(excluded: {nn} no alert, {na} matched entirely)")

struct = pd.DataFrame(rows)
struct.to_csv(f"{R}/table33_structural_regimes.csv", index=False)
print()
print(struct.pivot_table(index="ranking", columns="regime", values="lift")
            .reindex(RANKINGS + ["random atom order"])[["full","no_polycyclic","polycyclic_only"]]
            .round(3).to_string())

full             n=237  chance=0.4360  ceiling=1.9144  (excluded: 86 no alert, 13 matched entirely)
no_polycyclic    n=181  chance=0.2448  ceiling=2.8344  (excluded: 155 no alert, 0 matched entirely)
polycyclic_only  n=104  chance=0.6402  ceiling=1.5428  (excluded: 226 no alert, 6 matched entirely)

regime                full  no_polycyclic  polycyclic_only
ranking                                                   
degree               1.214          1.278            1.162
hetero_neighbours    1.163          1.679            0.812
degree_deviation     1.130          1.454            0.899
hetero_plus_degree   1.092          1.534            0.786
abs_formal_charge    1.067          1.170            0.992
heteroatom           1.059          1.513            0.744
hetero_or_neighbour  1.055          1.735            0.603
rarity               0.926          1.599            0.481
aromatic             0.982          0.519            1.297
in_ring              0.983          0.818         

In [6]:
# Pre-registered baseline against the panel maximum, per regime.
print("pre-registered baseline: degree\n")
for reg in REGIMES:
    s = struct[(struct.regime == reg) & (struct.ranking != "random atom order")]
    d = s[s.ranking == "degree"].iloc[0]
    top = s.loc[s.lift.idxmax()]
    print(f"  {reg:16s} degree {d.lift:.4f}   panel maximum {top.lift:.4f} ({top.ranking})"
          + ("" if top.ranking == "degree" else
             "   selection on the panel would raise the reference by "
             f"{top.lift - d.lift:+.4f}"))

pre-registered baseline: degree

  full             degree 1.2135   panel maximum 1.2135 (degree)
  no_polycyclic    degree 1.2782   panel maximum 1.7348 (hetero_or_neighbour)   selection on the panel would raise the reference by +0.4566
  polycyclic_only  degree 1.1615   panel maximum 1.2968 (aromatic)   selection on the panel would raise the reference by +0.1353


## 6. Replication on two further partitions

The degree result was obtained on the primary partition. Two further scaffold partitions
carry trained and untrained measurements from Part 5, on different molecules with different
alert coverage and therefore different attainable maxima.

In [7]:
rep = pd.read_csv(f"{R}/table28_attribution_replication.csv").set_index("partition")
rows = []
for sd in [PRIMARY_SPLIT_SEED] + REPLICATE_SEEDS:
    _, _, te = scaffold_split(sd)
    ev, ch, _, _ = build_eval(te, REGIMES["full"])
    CHANCE_CACHE[id(ev)] = ch
    ceil = ceiling(ev, ch, K_FRAC)
    v = scored(ev, descriptor(ev, "degree"), base=2000)
    key = f"scaffold_{sd}"
    if sd == PRIMARY_SPLIT_SEED:
        tg = pd.read_csv(f"{R}/table13_training_gain.csv").iloc[0]
        un, tr = tg.untrained_lift_mean_over_models, tg.trained_lift_mean_over_models
        n_un, n_tr = int(tg.n_untrained), int(tg.n_trained)
    else:
        un, tr = rep.loc[key, "untrained_lift_mean"], rep.loc[key, "trained_lift_mean"]
        n_un, n_tr = int(rep.loc[key, "n_untrained"]), int(rep.loc[key, "n_trained"])
    rows.append({"partition": key, "n_eval": len(ev), "chance": ch.mean(), "ceiling": ceil,
                 "degree": v.mean(), "degree_sd_tiebreak": v.std(ddof=1),
                 "untrained": un, "n_untrained": n_un,
                 "trained": tr, "n_trained": n_tr,
                 "degree_minus_trained": v.mean() - tr,
                 "degree_minus_untrained": v.mean() - un})
    print(f"  {key:14s} n={len(ev):3d}  ceiling {ceil:.3f}  degree {v.mean():.4f}  "
          f"untrained {un:.4f}  trained {tr:.4f}  -> degree "
          f"{'above' if v.mean() > tr else 'below'} trained")

repl = pd.DataFrame(rows)
repl.to_csv(f"{R}/table34_degree_replication.csv", index=False)
print()
n_above = int((repl.degree_minus_trained > 0).sum())
print(f"degree exceeds the trained model on {n_above} of {len(repl)} partitions")

  scaffold_101   n=237  ceiling 1.914  degree 1.2128  untrained 1.1156  trained 1.2420  -> degree below trained
  scaffold_201   n=243  ceiling 1.888  degree 1.2081  untrained 1.1266  trained 1.1880  -> degree above trained
  scaffold_202   n=226  ceiling 1.914  degree 1.2137  untrained 1.1404  trained 1.1514  -> degree above trained

degree exceeds the trained model on 2 of 3 partitions


## 7. Mechanism

Two rankings can reach the same aggregate agreement while succeeding on entirely different
molecules. Two measurements separate those possibilities. The first compares the two rankings
molecule by molecule. The second asks directly whether the trained model's attributions track
atom degree within each molecule, which is the mechanism proposed for the whole effect.

In [8]:
%%writefile /content/gnn_min.py
"""Minimal graph model definitions, matching Parts 2 to 6."""
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from rdkit import Chem
from torch_geometric.data import Data
from torch_geometric.nn import (GCNConv, GINConv, GINEConv, GATConv,
                                global_mean_pool, global_add_pool)

ATOM_DIM, BOND_DIM = 24, 6
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ATOM_LIST     = ['C','N','O','S','F','Cl','Br','I','P','B','Si','Se','other']
HYBRIDIZATION = ['SP','SP2','SP3','SP3D','SP3D2','other']

def one_hot(x, allowable):
    if x not in allowable: x = allowable[-1]
    return [int(x == a) for a in allowable]

def atom_features(atom):
    f  = one_hot(atom.GetSymbol(), ATOM_LIST)
    f += one_hot(str(atom.GetHybridization()), HYBRIDIZATION)
    f += [atom.GetDegree(), atom.GetFormalCharge(), int(atom.GetIsAromatic()),
          atom.GetTotalNumHs(), int(atom.IsInRing())]
    return f

def bond_features(bond):
    bt = bond.GetBondType()
    return [int(bt == Chem.BondType.SINGLE), int(bt == Chem.BondType.DOUBLE),
            int(bt == Chem.BondType.TRIPLE), int(bt == Chem.BondType.AROMATIC),
            int(bond.GetIsConjugated()), int(bond.IsInRing())]

def smiles_to_data(smiles, label):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None or mol.GetNumAtoms() == 0:
        return None
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float)
    ei, ea = [], []
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        bf = bond_features(b)
        ei += [[i, j], [j, i]]; ea += [bf, bf]
    if len(ei) == 0:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr  = torch.zeros((0, BOND_DIM), dtype=torch.float)
    else:
        edge_index = torch.tensor(ei, dtype=torch.long).t().contiguous()
        edge_attr  = torch.tensor(ea, dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr,
                y=torch.tensor([label], dtype=torch.float), smiles=smiles)

class GNN(nn.Module):
    def __init__(self, kind, in_dim=ATOM_DIM, edge_dim=BOND_DIM, hidden=128,
                 n_layers=3, dropout=0.2, heads=4, residual=True, use_edge_attr=True):
        super().__init__()
        self.kind, self.dropout, self.residual = kind, dropout, residual
        self.use_edge_attr = use_edge_attr and kind in ("gin", "gat")
        self.lin_in = nn.Linear(in_dim, hidden)
        self.convs, self.bns = nn.ModuleList(), nn.ModuleList()
        for _ in range(n_layers):
            if kind == "gcn":
                conv = GCNConv(hidden, hidden)
            elif kind == "gin":
                mlp = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(),
                                    nn.Linear(hidden, hidden))
                conv = (GINEConv(mlp, train_eps=True, edge_dim=edge_dim)
                        if self.use_edge_attr else GINConv(mlp, train_eps=True))
            elif kind == "gat":
                conv = GATConv(hidden, hidden // heads, heads=heads,
                               edge_dim=edge_dim if self.use_edge_attr else None)
            else:
                raise ValueError(kind)
            self.convs.append(conv); self.bns.append(nn.BatchNorm1d(hidden))
        self.head = nn.Sequential(nn.Linear(2 * hidden, hidden), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(hidden, 1))

    def forward(self, data):
        x, ei, ea, b = data.x, data.edge_index, data.edge_attr, data.batch
        x = self.lin_in(x)
        for conv, bn in zip(self.convs, self.bns):
            h = conv(x, ei, ea) if self.use_edge_attr else conv(x, ei)
            h = F.relu(bn(h))
            h = F.dropout(h, p=self.dropout, training=self.training)
            x = x + h if self.residual else h
        g = torch.cat([global_mean_pool(x, b), global_add_pool(x, b)], dim=1)
        return self.head(g).squeeze(-1)

def build_gnn(kind, hidden, n_layers, dropout, residual=True, heads=4, use_edge_attr=True):
    return GNN(kind, hidden=hidden, n_layers=n_layers, dropout=dropout,
               residual=residual, heads=heads, use_edge_attr=use_edge_attr).to(DEVICE)

class TensorView:
    def __init__(self, x, edge_index, edge_attr, batch):
        self.x, self.edge_index, self.edge_attr, self.batch = x, edge_index, edge_attr, batch

def set_seed(s):
    import random
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

Writing /content/gnn_min.py


In [9]:
import sys, torch
sys.path.insert(0, "/content")
from gnn_min import *
from torch_geometric.loader import DataLoader

N_IG_STEPS = 32
HP = json.load(open(f"{R}/hpo_gnn.json"))["gin"]

def integrated_gradients(model, graphs, n_steps=N_IG_STEPS, batch_size=64):
    model.eval()
    for p in model.parameters():
        p.requires_grad_(False)
    out = []
    for batch in DataLoader(graphs, batch_size=batch_size, shuffle=False):
        batch = batch.to(DEVICE)
        x0 = batch.x.clone()
        total = torch.zeros_like(x0)
        for alpha in torch.linspace(1.0 / n_steps, 1.0, n_steps):
            xa = (x0 * alpha).detach().requires_grad_(True)
            logit = model(TensorView(xa, batch.edge_index, batch.edge_attr, batch.batch))
            total += torch.autograd.grad(logit.sum(), xa)[0]
        sc = (x0 * (total / n_steps)).abs().sum(dim=1).detach().cpu().numpy()
        b = batch.batch.cpu().numpy()
        for g in range(batch.num_graphs):
            out.append(sc[b == g])
    return out

GRAPHS = [smiles_to_data(e["smiles"], 1.0) for e in EVAL]
gin_rep = build_gnn("gin", HP["hidden"], HP["n_layers"], HP["dropout"],
                    HP.get("residual", True))
gin_rep.load_state_dict(torch.load(f"{MODELS}/gin_seed{SEED}.pt",
                                   map_location=DEVICE)["state_dict"])
gin_rep.eval()
IG_SCORES = integrated_gradients(gin_rep, GRAPHS)
print(f"device {DEVICE}; Integrated Gradients computed for {len(IG_SCORES)} molecules")

device cpu; Integrated Gradients computed for 237 molecules


In [10]:
from scipy.stats import spearmanr, pearsonr

DEG = descriptor(EVAL, "degree")

# (a) per-molecule agreement between the degree ranking and the trained models
deg_ov = np.nanmean([overlap(EVAL, DEG, K_FRAC, 3000 + j) for j in range(N_TIEBREAK)], axis=0)
tr_ov  = np.nanmean(np.load(f"{EXPL}/trained_overlaps.npz")["overlaps"], axis=0)
un_ov  = np.nanmean(np.load(f"{EXPL}/untrained_overlaps_gin.npy"), axis=0)
m = ~(np.isnan(deg_ov) | np.isnan(tr_ov) | np.isnan(un_ov))
r_dt = pearsonr(deg_ov[m], tr_ov[m]); s_dt = spearmanr(deg_ov[m], tr_ov[m])
r_du = pearsonr(deg_ov[m], un_ov[m])
r_tu = pearsonr(tr_ov[m], un_ov[m])

# (b) within-molecule association between the attributions and atom degree
sp = np.array([spearmanr(ig, d).correlation if len(set(d)) > 1 else np.nan
               for ig, d in zip(IG_SCORES, DEG)])
sp = sp[~np.isnan(sp)]

mech = pd.DataFrame([
    {"quantity": "per-molecule overlap, degree vs trained", "n": int(m.sum()),
     "pearson_r": r_dt[0], "p": r_dt[1], "spearman_r": s_dt.correlation},
    {"quantity": "per-molecule overlap, degree vs untrained", "n": int(m.sum()),
     "pearson_r": r_du[0], "p": r_du[1], "spearman_r": np.nan},
    {"quantity": "per-molecule overlap, trained vs untrained", "n": int(m.sum()),
     "pearson_r": r_tu[0], "p": r_tu[1], "spearman_r": np.nan},
    {"quantity": "within-molecule Spearman, IG vs atom degree", "n": len(sp),
     "pearson_r": np.nan, "p": np.nan, "spearman_r": sp.mean()},
])
mech.to_csv(f"{R}/table35_mechanism.csv", index=False)
print(mech.round(4).to_string(index=False))
print()
print(f"within-molecule Spearman(IG, degree): mean {sp.mean():+.3f}, "
      f"median {np.median(sp):+.3f}, positive in {100*(sp>0).mean():.1f}% of molecules")
np.save(f"{EXPL}/spearman_ig_degree.npy", sp)
np.save(f"{EXPL}/degree_overlaps_primary.npy", deg_ov)

                                   quantity   n  pearson_r   p  spearman_r
    per-molecule overlap, degree vs trained 237     0.8236 0.0      0.8305
  per-molecule overlap, degree vs untrained 237     0.9431 0.0         NaN
 per-molecule overlap, trained vs untrained 237     0.7591 0.0         NaN
within-molecule Spearman, IG vs atom degree 237        NaN NaN      0.5271

within-molecule Spearman(IG, degree): mean +0.527, median +0.564, positive in 97.5% of molecules


## 8. Agreement across retained fractions

The comparison is reported at a retained fraction of 30%. Repeating it across fractions shows
whether the ordering depends on that choice. The untrained band is recomputed here with the
full 500 draws, so that a single figure for the percentile is used throughout; the earlier
sweep used 100 draws and reported a different value for the same quantity.

In [11]:
t0 = time.time()
un_scores_k = {k: np.empty(N_UNTRAINED) for k in K_GRID}
un_raw = {k: np.empty((N_UNTRAINED, len(EVAL))) for k in K_GRID}
for j in range(N_UNTRAINED):
    set_seed(50_000 + j)
    m_ = build_gnn("gin", HP["hidden"], HP["n_layers"], HP["dropout"], HP.get("residual", True))
    sc = integrated_gradients(m_, GRAPHS)
    for k in K_GRID:
        ov = overlap(EVAL, sc, k, 50_000 + j)
        un_raw[k][j] = ov
        un_scores_k[k][j] = lift(ov, CHANCE)
    del m_
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    if (j + 1) % 100 == 0:
        print(f"  {j+1}/{N_UNTRAINED} untrained draws ({(time.time()-t0)/60:.1f} min)")
np.savez_compressed(f"{EXPL}/untrained_overlaps_by_k.npz",
                    **{f"k{int(k*100)}": un_raw[k] for k in K_GRID})

t17 = pd.read_csv(f"{R}/table17_topk_sensitivity.csv")
rows = []
for k in K_GRID:
    ev_k, ch_k = EVAL, CHANCE
    ceil_k = ceiling(ev_k, ch_k, k)
    d = scored(ev_k, DEG, k_frac=k, base=4000)
    u = un_scores_k[k]
    ig = t17[(t17.source == "GIN (Integrated Gradients)") & (t17.k_fraction == k)]
    ts = t17[(t17.source.str.contains("TreeSHAP")) & (t17.k_fraction == k)]
    rows.append({"k_fraction": k, "ceiling": ceil_k,
                 "degree": d.mean(), "degree_sd_tiebreak": d.std(ddof=1),
                 "untrained_mean": u.mean(), "untrained_sd": u.std(ddof=1),
                 "untrained_p2_5": np.percentile(u, 2.5),
                 "untrained_p97_5": np.percentile(u, 97.5),
                 "trained_representative": float(ig.lift.iloc[0]) if len(ig) else np.nan,
                 "treeshap": float(ts.lift.iloc[0]) if len(ts) else np.nan})
sweep = pd.DataFrame(rows)
sweep.to_csv(f"{R}/table36_ktop_sweep.csv", index=False)
print()
print(sweep.round(4).to_string(index=False))
print()
print("the 97.5th percentile of the untrained distribution, 500 draws:")
for _, r in sweep.iterrows():
    print(f"  k={r.k_fraction:.2f}  untrained p97.5 {r.untrained_p97_5:.4f}   "
          f"trained {r.trained_representative:.4f}   degree {r.degree:.4f}")

  100/500 untrained draws (1.9 min)
  200/500 untrained draws (3.8 min)
  300/500 untrained draws (5.7 min)
  400/500 untrained draws (7.6 min)
  500/500 untrained draws (9.5 min)

 k_fraction  ceiling  degree  degree_sd_tiebreak  untrained_mean  untrained_sd  untrained_p2_5  untrained_p97_5  trained_representative  treeshap
        0.1   2.2805  1.2001              0.0354          1.1764        0.1001          0.9786           1.3541                  1.3619    1.8015
        0.2   2.1292  1.2172              0.0194          1.1452        0.0764          0.9896           1.2943                  1.2842    1.6949
        0.3   1.9144  1.2136              0.0141          1.1205        0.0612          0.9952           1.2314                  1.2265    1.5422
        0.4   1.7384  1.1677              0.0082          1.0917        0.0487          0.9972           1.1832                  1.1724    1.4212
        0.5   1.5910  1.1079              0.0082          1.0657        0.0398          0

## 9. Control distributions

Three controls were reported earlier as means, and the conclusion drawn from them was that
the effect requires message passing. Means are not enough: the draws behind them have
intervals, and whether those intervals separate is the question the argument actually rests
on. Each control is recomputed here with its draws retained, and compared with the untrained
network at the level of distributions rather than points.

In [12]:
from rdkit import DataStructs
from rdkit.Chem import rdFingerprintGenerator as rfg
FP_BITS, FP_RADIUS = 2048, 2
morgan = rfg.GetMorganGenerator(radius=FP_RADIUS, fpSize=FP_BITS)

def fp_bitinfo(mol):
    ao = rfg.AdditionalOutput(); ao.AllocateBitInfoMap()
    morgan.GetFingerprint(mol, additionalOutput=ao)
    return ao.GetBitInfoMap()

def bit_atoms(mol, atom_idx, radius):
    if radius == 0:
        return {atom_idx}
    env = Chem.FindAtomEnvironmentOfRadiusN(mol, radius, atom_idx)
    s = {atom_idx}
    for b in env:
        bd = mol.GetBondWithIdx(b); s.add(bd.GetBeginAtomIdx()); s.add(bd.GetEndAtomIdx())
    return s

BITINFO = [fp_bitinfo(e["mol"]) for e in EVAL]
ATOMFEAT = [np.array([atom_features(a) for a in e["mol"].GetAtoms()], dtype=float) for e in EVAL]

N_CTRL = 500
def control_draws(kind, n=N_CTRL):
    vals = np.empty(n); raw = np.empty((n, len(EVAL)))
    for j in range(n):
        rng = np.random.default_rng(60_000 + j)
        if kind == "random_atom_order":
            sc = [rng.random(e["n_atoms"]) for e in EVAL]
        elif kind == "random_feature_weighting":
            w = rng.random(ATOM_DIM)
            sc = [f @ w for f in ATOMFEAT]
        elif kind == "random_through_bit_map":
            vec = rng.normal(size=FP_BITS)
            sc = []
            for e, bi in zip(EVAL, BITINFO):
                s = np.zeros(e["n_atoms"])
                for bit, envs in bi.items():
                    v = vec[bit]
                    if len(envs) == 0: continue
                    for ai, rad in envs:
                        atoms = bit_atoms(e["mol"], ai, rad)
                        share = v / (len(envs) * len(atoms))
                        for a in atoms: s[a] += share
                sc.append(s)
        else:
            raise ValueError(kind)
        ov = overlap(EVAL, sc, K_FRAC, 60_000 + j)
        raw[j] = ov; vals[j] = lift(ov, CHANCE)
    return vals, raw

CTRLS = {}
for kind in ["random_atom_order", "random_feature_weighting", "random_through_bit_map"]:
    t0 = time.time()
    CTRLS[kind], raw = control_draws(kind)
    np.save(f"{EXPL}/control_{kind}.npy", raw)
    print(f"  {kind:26s} {CTRLS[kind].mean():.4f} "
          f"[{np.percentile(CTRLS[kind],2.5):.4f}, {np.percentile(CTRLS[kind],97.5):.4f}]  "
          f"({time.time()-t0:.0f} s)")

un_lift = un_scores_k[K_FRAC]
rows = []
for kind, v in list(CTRLS.items()) + [("untrained_gin", un_lift)]:
    row = {"control": kind, "n": len(v), "mean": v.mean(), "sd": v.std(ddof=1),
           "p2_5": np.percentile(v, 2.5), "p97_5": np.percentile(v, 97.5)}
    if kind != "untrained_gin":
        diff = un_lift.mean() - v.mean()
        pooled = np.sqrt((un_lift.var(ddof=1) + v.var(ddof=1)) / 2)
        sup = np.mean(un_lift[:, None] > v[None, :])
        pool = np.concatenate([un_lift, v]); n1 = len(un_lift)
        rng = np.random.default_rng(SEED)
        null = np.array([(lambda p: p[:n1].mean() - p[n1:].mean())(rng.permutation(pool))
                         for _ in range(10000)])
        row.update({"untrained_minus_control": diff, "cohens_d": diff / pooled,
                    "superiority_prob": sup,
                    "permutation_p": float((np.abs(null) >= abs(diff)).mean())})
    rows.append(row)
ctrl = pd.DataFrame(rows)
ctrl.to_csv(f"{R}/table37_control_distributions.csv", index=False)
print()
print(ctrl.round(4).to_string(index=False))

  random_atom_order          0.9999 [0.9557, 1.0418]  (1 s)
  random_feature_weighting   0.9724 [0.7544, 1.2169]  (1 s)
  random_through_bit_map     1.0052 [0.8276, 1.1859]  (64 s)

                 control   n   mean     sd   p2_5  p97_5  untrained_minus_control  cohens_d  superiority_prob  permutation_p
       random_atom_order 500 0.9999 0.0225 0.9557 1.0418                   0.1206    2.6155            0.9625            0.0
random_feature_weighting 500 0.9724 0.1555 0.7544 1.2169                   0.1481    1.2532            0.7526            0.0
  random_through_bit_map 500 1.0052 0.0963 0.8276 1.1859                   0.1153    1.4288            0.8388            0.0
           untrained_gin 500 1.1205 0.0612 0.9952 1.2314                      NaN       NaN               NaN            NaN


## 10. What predicts the effect of training

An earlier regression took the raw overlap as its target while including alert coverage as a
predictor. Chance overlap is alert coverage, so that specification was close to tautological
and the coefficient it produced carried no information. The target here is the difference
between the trained and untrained overlaps for the same molecule, which is the quantity the
claim concerns. Collinearity among the predictors is reported alongside.

In [13]:
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

def mol_features(ev):
    rows = []
    for e in ev:
        ats = list(e["mol"].GetAtoms())
        deg = np.array([a.GetDegree() for a in ats], dtype=float)
        rows.append({"alert_coverage": len(e["tox"]) / e["n_atoms"],
                     "n_atoms": e["n_atoms"],
                     "heteroatom_fraction": np.mean([a.GetSymbol() in HETERO for a in ats]),
                     "aromatic_fraction": np.mean([a.GetIsAromatic() for a in ats]),
                     "mean_degree": deg.mean(), "degree_spread": deg.std()})
    return pd.DataFrame(rows)

X = mol_features(EVAL)
Xc = sm.add_constant(X)
vif = pd.DataFrame({"predictor": X.columns,
                    "VIF": [variance_inflation_factor(Xc.values, i + 1)
                            for i in range(X.shape[1])]})
print("variance inflation factors")
print(vif.round(2).to_string(index=False))
print()
print("pairwise correlations")
print(X.corr().round(2).to_string())

y = tr_ov - un_ov
mask = ~np.isnan(y)
model = sm.OLS(y[mask], sm.add_constant(X[mask])).fit()
print()
print(model.summary().tables[1])
print(f"R^2 = {model.rsquared:.4f}   n = {int(mask.sum())}")

reg = pd.DataFrame({"predictor": ["const"] + list(X.columns),
                    "beta": model.params.values, "p": model.pvalues.values})
reg["r_squared"] = model.rsquared
reg = reg.merge(vif, on="predictor", how="left")
reg.to_csv(f"{R}/table38_training_effect_regression.csv", index=False)
vif.to_csv(f"{R}/table38b_vif.csv", index=False)

variance inflation factors
          predictor  VIF
     alert_coverage 1.30
            n_atoms 1.31
heteroatom_fraction 2.59
  aromatic_fraction 2.07
        mean_degree 2.21
      degree_spread 2.15

pairwise correlations
                     alert_coverage  n_atoms  heteroatom_fraction  aromatic_fraction  mean_degree  degree_spread
alert_coverage                 1.00    -0.09                -0.44               0.28         0.40          -0.27
n_atoms                       -0.09     1.00                -0.13               0.10         0.10           0.26
heteroatom_fraction           -0.44    -0.13                 1.00              -0.54        -0.70           0.51
aromatic_fraction              0.28     0.10                -0.54               1.00         0.54          -0.59
mean_degree                    0.40     0.10                -0.70               0.54         1.00          -0.34
degree_spread                 -0.27     0.26                 0.51              -0.59        -0.34

## 11. Uncertainty on three axes

The interval reported for the effect of training came from resampling molecules alone. The
claim concerns models, and a third source of variation is the partition. Molecules and models
are resampled together below; the partition axis is read from the three partitions in
Section 6 and reported beside it, since three partitions do not support a resampling
estimate.

In [14]:
TR = np.load(f"{EXPL}/trained_overlaps.npz")["overlaps"]
UN = np.load(f"{EXPL}/untrained_overlaps_gin.npy")
keep = ~(np.isnan(TR).all(0) | np.isnan(UN).all(0))
TRk, UNk, CHk = TR[:, keep], UN[:, keep], CHANCE[keep]
nM = TRk.shape[1]
print(f"trained {TR.shape}, untrained {UN.shape}, molecules retained {nM}")

rng = np.random.default_rng(SEED)
def boot(axis, B=N_BOOT):
    t_mean, u_mean = np.nanmean(TRk, 0), np.nanmean(UNk, 0)
    out = np.empty(B)
    for b in range(B):
        mi = rng.integers(0, nM, nM)
        if axis == "molecule":
            out[b] = (np.nanmean(t_mean[mi]) - np.nanmean(u_mean[mi])) / np.mean(CHk[mi])
        else:
            ti = rng.integers(0, TRk.shape[0], TRk.shape[0])
            ui = rng.integers(0, UNk.shape[0], UNk.shape[0])
            out[b] = (np.nanmean(TRk[np.ix_(ti, mi)]) -
                      np.nanmean(UNk[np.ix_(ui, mi)])) / np.mean(CHk[mi])
    return out

rows = []
for axis in ["molecule", "molecule_and_model"]:
    v = boot(axis)
    lo, hi = np.percentile(v, [2.5, 97.5])
    rows.append({"axis": axis, "delta_lift": v.mean(), "ci_low": lo, "ci_high": hi,
                 "width": hi - lo})
    print(f"  {axis:20s} {v.mean():+.4f}  [{lo:+.4f}, {hi:+.4f}]  width {hi-lo:.4f}")

frac = repl.copy()
frac["pretraining_fraction"] = (frac.untrained - 1) / (frac.trained - 1)
rows.append({"axis": "partition (three observed)",
             "delta_lift": np.nan, "ci_low": np.nan, "ci_high": np.nan, "width": np.nan})
unc = pd.DataFrame(rows)
unc.to_csv(f"{R}/table39_uncertainty_axes.csv", index=False)
frac[["partition", "untrained", "trained", "pretraining_fraction"]].to_csv(
    f"{R}/table39b_pretraining_fraction.csv", index=False)
print()
print(f"widening from adding the model axis: "
      f"{unc.iloc[1].width / unc.iloc[0].width:.2f}x")
print()
print("pre-training fraction across the three partitions:")
print(frac[["partition", "untrained", "trained", "pretraining_fraction"]].round(4)
          .to_string(index=False))

t_lift = np.nanmean(TRk, 0).mean() / CHk.mean()
u_lift = np.nanmean(UNk, 0).mean() / CHk.mean()
tr_per_model = np.nanmean(TRk, 1) / CHk.mean()
un_per_model = np.nanmean(UNk, 1) / CHk.mean()
deg_point = repl[repl.partition == f"scaffold_{PRIMARY_SPLIT_SEED}"].degree.iloc[0]
sup_tr_deg = float((tr_per_model > deg_point).mean())
sup_tr_un = float(np.mean(tr_per_model[:, None] > un_per_model[None, :]))
pd.DataFrame([{"trained_lift": t_lift, "untrained_lift": u_lift, "degree_point": deg_point,
               "P_trained_exceeds_degree": sup_tr_deg,
               "P_trained_exceeds_untrained": sup_tr_un,
               "n_trained": TRk.shape[0], "n_untrained": UNk.shape[0]}]).to_csv(
    f"{R}/table40_superiority.csv", index=False)
print()
print(f"the trained distribution exceeds the degree value in {100*sup_tr_deg:.1f}% of draws")
print(f"a trained model exceeds an untrained one in {100*sup_tr_un:.1f}% of pairs "
      f"(one in {1/(1-sup_tr_un):.1f} goes the other way)")

trained (50, 237), untrained (500, 237), molecules retained 237
  molecule             +0.1275  [+0.0573, +0.2023]  width 0.1450
  molecule_and_model   +0.1271  [+0.0507, +0.2087]  width 0.1580

widening from adding the model axis: 1.09x

pre-training fraction across the three partitions:
   partition  untrained  trained  pretraining_fraction
scaffold_101     1.1156   1.2420                0.4778
scaffold_201     1.1266   1.1880                0.6735
scaffold_202     1.1404   1.1514                0.9269

the trained distribution exceeds the degree value in 62.0% of draws
a trained model exceeds an untrained one in 88.6% of pairs (one in 8.8 goes the other way)


## 12. Dependence between partitions

The family comparison treats fifteen scaffold partitions as repeated observations. Whether
they are is measured here, for the splitter used throughout and for a variant that orders
scaffold groups at random rather than by descending size. If the dependence follows from the
ordering rather than from scaffold partitioning itself, it is a property of the
implementation and can be removed.

In [15]:
def test_sets(seeds, kind, order="size"):
    out = {}
    for sd in seeds:
        tr, va, te = (scaffold_split(sd, order=order) if kind == "scaffold"
                      else random_split(sd))
        te_s = set(te["canonical"])
        tr_scaf = {SCAF[df.index.get_loc(i)] for i in tr.index}
        te_scaf = {SCAF[df.index.get_loc(i)] for i in te.index}
        out[sd] = (te_s, tr_scaf, te_scaf)
    return out

rows = []
for label, kind, order in [("scaffold, size-ordered", "scaffold", "size"),
                           ("scaffold, random order", "scaffold", "random"),
                           ("random", "random", None)]:
    ts = test_sets(BENCH_SEEDS, kind, order or "size")
    js = []
    for i, a in enumerate(BENCH_SEEDS):
        for b in BENCH_SEEDS[i+1:]:
            A, B = ts[a][0], ts[b][0]
            js.append(len(A & B) / len(A | B))
    J = float(np.mean(js))
    shared = 2 * J / (1 + J)
    k = len(BENCH_SEEDS)
    n_eff = k / (1 + (k - 1) * shared)
    union = set().union(*[ts[s][0] for s in BENCH_SEEDS])
    leak = [100 * len(ts[s][2] & ts[s][1]) / max(len(ts[s][2]), 1) for s in BENCH_SEEDS]
    rows.append({"scheme": label, "n_partitions": k, "mean_jaccard": J,
                 "min_jaccard": float(np.min(js)), "max_jaccard": float(np.max(js)),
                 "expected_if_independent": 0.1 / 1.9,
                 "shared_fraction": shared, "n_effective": n_eff,
                 "union_compounds": len(union), "per_partition": len(ts[BENCH_SEEDS[0]][0]),
                 "pool": len(df),
                 "test_scaffolds_in_training_pct": float(np.mean(leak))})
    print(f"  {label:24s} Jaccard {J:.4f}  N_eff {n_eff:.2f}/{k}  "
          f"union {len(union)} of {len(df)}  leakage {np.mean(leak):.1f}%")

part = pd.DataFrame(rows)
part.to_csv(f"{R}/table41_partition_dependence.csv", index=False)
print()
print("Two independent tenth-samples of the same pool would overlap at a Jaccard of 0.053.")
print("N_eff is the design-effect approximation k / (1 + (k-1) * shared fraction).")

  scaffold, size-ordered   Jaccard 0.6335  N_eff 1.26/15  union 937 of 7255  leakage 0.0%
  scaffold, random order   Jaccard 0.0515  N_eff 6.32/15  union 5965 of 7255  leakage 0.0%
  random                   Jaccard 0.0528  N_eff 6.24/15  union 5773 of 7255  leakage 67.6%

Two independent tenth-samples of the same pool would overlap at a Jaccard of 0.053.
N_eff is the design-effect approximation k / (1 + (k-1) * shared fraction).


## 13. Summary

In [16]:
print("=== structural baselines, primary partition, k = 0.30 ===")
s = struct[(struct.regime == "full")].sort_values("lift", ascending=False)
for _, r in s.head(6).iterrows():
    print(f"  {r.ranking:22s} {r.lift:.4f}   {r.pct_of_ceiling:5.1f}% of ceiling")
print(f"  {'untrained GIN':22s} {un_scores_k[K_FRAC].mean():.4f}")
print(f"  {'trained GIN, IG':22s} {t_lift:.4f}")

print("\n=== replication ===")
print(repl[["partition", "degree", "untrained", "trained"]].round(4).to_string(index=False))

print("\n=== mechanism ===")
print(f"  per-molecule overlap, degree vs trained : r = {r_dt[0]:+.3f}")
print(f"  within-molecule Spearman(IG, degree)    : {sp.mean():+.3f}")

print("\n=== uncertainty ===")
print(f"  molecule axis only : width {unc.iloc[0].width:.4f}")
print(f"  molecule and model : width {unc.iloc[1].width:.4f}")

print("\n=== partitions ===")
print(part[["scheme", "mean_jaccard", "n_effective", "union_compounds"]].round(3)
          .to_string(index=False))

print("\nWritten to", R)
for f in ["table33_structural_regimes.csv", "table34_degree_replication.csv",
          "table35_mechanism.csv", "table36_ktop_sweep.csv",
          "table37_control_distributions.csv", "table38_training_effect_regression.csv",
          "table38b_vif.csv", "table39_uncertainty_axes.csv",
          "table39b_pretraining_fraction.csv", "table40_superiority.csv",
          "table41_partition_dependence.csv"]:
    print("  -", f)

import shutil
from google.colab import files
shutil.make_archive("/content/part8_results", "zip", R)
files.download("/content/part8_results.zip")

=== structural baselines, primary partition, k = 0.30 ===
  degree                 1.2135    23.4% of ceiling
  hetero_neighbours      1.1632    17.8% of ceiling
  degree_deviation       1.1295    14.2% of ceiling
  hetero_plus_degree     1.0922    10.1% of ceiling
  abs_formal_charge      1.0669     7.3% of ceiling
  heteroatom             1.0593     6.5% of ceiling
  untrained GIN          1.1205
  trained GIN, IG        1.2420

=== replication ===
   partition  degree  untrained  trained
scaffold_101  1.2128     1.1156   1.2420
scaffold_201  1.2081     1.1266   1.1880
scaffold_202  1.2137     1.1404   1.1514

=== mechanism ===
  per-molecule overlap, degree vs trained : r = +0.824
  within-molecule Spearman(IG, degree)    : +0.527

=== uncertainty ===
  molecule axis only : width 0.1450
  molecule and model : width 0.1580

=== partitions ===
                scheme  mean_jaccard  n_effective  union_compounds
scaffold, size-ordered         0.634        1.265              937
scaffold,

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>